In [25]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

In [2]:
# Load Parquet file
rule_of_law_df = pd.read_parquet('../data/image_data/wb_ruleoflaw.parquet')
share_of_industry_df = pd.read_parquet('../data/image_data/Share_of_industry.parquet')
tax_revenue_df = pd.read_parquet('../data/image_data/Tax_revenue.parquet')
inflation_df = pd.read_parquet('../data/image_data/Inflation.parquet')
edu_spending_df = pd.read_parquet('../data/image_data/Education_spending.parquet')
govermment_effective_df = pd.read_parquet('../data/image_data/wb_government_effectiveness.parquet')

In [5]:
rule_of_law_df = rule_of_law_df.reset_index().rename(columns={"index": "Year"})
share_of_industry_df = share_of_industry_df.reset_index().rename(columns={"index": "Year"})
tax_revenue_df = tax_revenue_df.reset_index().rename(columns={"index": "Year"})
inflation_df = inflation_df.reset_index().rename(columns={"index": "Year"})
edu_spending_df = edu_spending_df.reset_index().rename(columns={"index": "Year"})
govermment_effective_df = govermment_effective_df.reset_index().rename(columns={"index": "Year"})

In [6]:
name_df = [
    'rule_of_law_df', 'share_of_industry_df', 'tax_revenue_df', 'inflation_df', 'edu_spending_df', 'govermment_effective_df'
]

In [11]:
list_df = [
    rule_of_law_df, share_of_industry_df, tax_revenue_df, inflation_df, edu_spending_df, govermment_effective_df
]

In [9]:
def remove_columns(df, columns_to_remove):
    """
    Removes specific columns from a DataFrame.

    Parameters:
    - df: Original DataFrame
    - columns_to_remove: list of column names to remove

    Returns:
    - DataFrame without the specified columns
    """
    return df.drop(columns=columns_to_remove, errors='ignore')

In [10]:
# countries to delete
delete_countries = ['China', 'Morocco', 'Macedonia', 'Indonesia']

rule_of_law_df = remove_columns(rule_of_law_df, delete_countries)
share_of_industry_df = remove_columns(share_of_industry_df, delete_countries)
tax_revenue_df = remove_columns(tax_revenue_df, delete_countries)
inflation_df = remove_columns(inflation_df, delete_countries)
edu_spending_df = remove_columns(edu_spending_df, delete_countries)
govermment_effective_df = remove_columns(govermment_effective_df, delete_countries)

In [12]:
def check_columns_in_filenames(dataframes, df_names, path):
    """
    Checks whether column names from each DataFrame exist in the filenames of a given path,
    accounting for differences like '-' in column names vs '_' in filenames.

    Parameters:
    - dataframes: list of pandas DataFrames
    - df_names: list of names corresponding to each DataFrame
    - path: directory path containing files

    Prints:
    - For each DataFrame, its name followed by the list of column names
      that do NOT appear in any of the filenames in the specified path.
    """
    try:
        filenames = os.listdir(path)
    except FileNotFoundError:
        print(f"❌ Path not found: {path}")
        return

    # Normalize filenames: lowercase and replace underscores with hyphens
    normalized_filenames = [f.lower().replace("_", "-") for f in filenames]

    for df, name in zip(dataframes, df_names):
        missing_columns = []
        for col in df.columns:
            normalized_col = col.lower().replace("_", "-")
            if not any(normalized_col in fname for fname in normalized_filenames):
                missing_columns.append(col)

        print(f"📊 DataFrame: {name}")
        if missing_columns:
            print(f"   ❌ Columns not found in filenames: {missing_columns}")
        else:
            print("   ✅ All columns found in filenames")

In [14]:
check_columns_in_filenames(list_df, name_df, '../data/by_country')

📊 DataFrame: rule_of_law_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: share_of_industry_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: tax_revenue_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: inflation_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: edu_spending_df
   ❌ Columns not found in filenames: ['Year']
📊 DataFrame: govermment_effective_df
   ❌ Columns not found in filenames: ['Year']


In [15]:
def update_country_parquets_by_name_match(macro_dict, output_path):
    os.makedirs(output_path, exist_ok=True)
    existing_files = [f for f in os.listdir(output_path) if f.endswith('.parquet')]

    for indicator, df in macro_dict.items():
        for country in df.columns:
            if country == 'Year':
                continue

            # Solo procesar si el nombre del país contiene "-"
            if "-" not in country:
                continue

            # Normalizar nombre del país para emparejar con archivos
            normalized_country = country.replace("-", "_")

            # Buscar archivo que contenga el nombre normalizado
            matched_file = next(
                (f for f in existing_files if normalized_country in f),
                None
            )

            if matched_file:
                file_path = os.path.join(output_path, matched_file)
                existing_df = pd.read_parquet(file_path)

                # Extraer columna del país y renombrar con el indicador
                country_data = df[['Year', country]].copy()
                country_data.rename(columns={country: indicator}, inplace=True)

                # Actualizar archivo existente
                updated_df = pd.merge(existing_df, country_data, on='Year', how='outer')
                updated_df.sort_values(by='Year', inplace=True)
                updated_df.to_parquet(file_path, index=False)
            else:
                print(f"⚠️ No match found for country: {country}")

In [16]:
macro_dict = {
    'rule_of_law_df': rule_of_law_df,
    'share_of_industry_df': share_of_industry_df,
    'tax_revenue_df': tax_revenue_df,
    'inflation_df': inflation_df,
    'edu_spending_df': edu_spending_df,
    'govermment_effective_df': govermment_effective_df
}

In [17]:
update_country_parquets_by_name_match(macro_dict, '../data/by_country')

In [27]:
czech_df_01 = pd.read_parquet('../data/by_country/Czech_Republic_macro_profile.parquet')

In [28]:
czech_df_01

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
0,2007,NaN,NaN,NaN,NaN,NaN,NaN,286.0300,NaN,964.1600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2008,NaN,NaN,NaN,NaN,NaN,NaN,291.7725,NaN,1009.3900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,NaN,NaN,NaN,NaN,NaN,NaN,268.1400,NaN,986.3775,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,NaN,NaN,NaN,NaN,NaN,NaN,267.4050,NaN,1008.2400,NaN,NaN,99.9975,NaN,NaN,NaN,NaN,NaN,NaN
4,2011,NaN,NaN,NaN,NaN,NaN,NaN,267.2050,NaN,1023.8400,NaN,NaN,100.0250,NaN,NaN,NaN,NaN,NaN,NaN
5,2012,NaN,NaN,NaN,NaN,NaN,NaN,261.2600,97.975000,1029.5975,NaN,NaN,98.5550,NaN,NaN,NaN,NaN,NaN,NaN
6,2013,NaN,NaN,NaN,NaN,NaN,920.626667,258.2225,99.341667,1042.2525,NaN,NaN,98.5800,NaN,NaN,NaN,NaN,NaN,NaN
7,2014,NaN,NaN,NaN,NaN,NaN,1182.200000,270.7900,99.750000,1094.4975,2072.538333,NaN,100.9950,NaN,NaN,NaN,NaN,3.67,NaN
8,2015,NaN,101.3,45.25,NaN,220.0425,1431.200833,298.6050,100.008333,1162.9525,2202.911667,NaN,105.0125,NaN,NaN,14.77,NaN,5.75,NaN
9,2016,5.35,101.8,45.18,73.57,228.7500,1902.431667,294.3300,100.658333,1210.7600,2356.845833,3.966667,112.5225,0.99,NaN,14.91,NaN,5.55,1.01


In [31]:
iceland_df = pd.read_parquet("../data/by_country/Iceland_macro_profile.parquet")
iceland_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,NaN,103.3,18.57,93.90,209.5500,917.633333,156.7075,103.058333,745.9925,2802.910000,6.433333,220.8375,1.75,20.73,22.40,2.8,8.58,1.48
14,2021,NaN,103.5,18.57,93.94,228.6000,882.272500,187.2875,106.840833,832.8850,3151.795000,6.016667,249.4875,1.71,21.05,21.41,4.4,8.22,1.59
15,2022,NaN,101.3,18.57,93.99,252.8350,879.947500,234.4025,112.959167,987.4575,3477.705833,3.591667,302.4725,1.70,21.87,22.52,8.3,7.14,1.56
16,2023,NaN,NaN,NaN,94.04,275.4025,790.423333,269.4750,121.964167,1092.9225,3770.335000,3.375000,322.8275,1.72,20.12,NaN,8.7,NaN,1.56
17,2024,NaN,NaN,NaN,NaN,298.1075,883.040000,305.2450,127.466667,1149.6000,4023.268333,3.375000,350.1350,NaN,19.40,NaN,5.9,NaN,NaN


In [32]:
iceland_labor_force = {
    2015: 83,
    2016: 83.8,
    2017: 82.9,
    2018: 81.8,
    2019: 81.2,
    2020: 79,
    2021: 73.5,
    2022: 74.6,
    2023: 75
}

In [33]:
iceland_df["labor_force"] = iceland_df["Year"].map(iceland_labor_force)

In [34]:
iceland_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,79.0,103.3,18.57,93.90,209.5500,917.633333,156.7075,103.058333,745.9925,2802.910000,6.433333,220.8375,1.75,20.73,22.40,2.8,8.58,1.48
14,2021,73.5,103.5,18.57,93.94,228.6000,882.272500,187.2875,106.840833,832.8850,3151.795000,6.016667,249.4875,1.71,21.05,21.41,4.4,8.22,1.59
15,2022,74.6,101.3,18.57,93.99,252.8350,879.947500,234.4025,112.959167,987.4575,3477.705833,3.591667,302.4725,1.70,21.87,22.52,8.3,7.14,1.56
16,2023,75.0,NaN,NaN,94.04,275.4025,790.423333,269.4750,121.964167,1092.9225,3770.335000,3.375000,322.8275,1.72,20.12,NaN,8.7,NaN,1.56
17,2024,NaN,NaN,NaN,NaN,298.1075,883.040000,305.2450,127.466667,1149.6000,4023.268333,3.375000,350.1350,NaN,19.40,NaN,5.9,NaN,NaN


In [36]:
iceland_df.to_parquet('../data/by_country/Iceland_macro_profile.parquet', index=False)

In [37]:
israel_df = pd.read_parquet("../data/by_country/Israel_macro_profile.parquet")
israel_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,4.21,107.6,NaN,92.59,82.7200,151.702500,80.5050,87.858333,355.5975,987.480833,4.308333,162.9000,0.94,19.44,21.91,-0.6,6.48,1.05
14,2021,4.28,105.1,NaN,92.67,87.6275,198.805000,94.0525,89.191667,395.5800,1082.482500,5.050000,176.6475,0.91,18.84,24.62,1.5,6.18,1.25
15,2022,4.47,109.4,NaN,92.76,92.3250,197.190833,109.8650,93.108333,441.1000,1264.530000,3.766667,207.1650,0.94,19.43,25.01,4.4,6.46,1.24
16,2023,4.57,NaN,NaN,92.85,105.8425,200.111667,114.3925,97.041667,469.5925,1378.055833,3.466667,214.2300,0.78,18.50,NaN,4.2,NaN,1.15
17,2024,NaN,NaN,NaN,NaN,126.7375,212.894167,113.4075,100.008333,499.5200,1448.846667,3.466667,224.5400,NaN,17.35,NaN,3.1,NaN,NaN


In [39]:
agr_land = {
    2015: 24.7,
    2016: 25.5,
    2017: 29.8,
    2018: 29.6,
    2019: 29.5,
    2020: 29.9,
    2021: 29.7,
}

In [40]:
israel_df["agr_land"] = israel_df["Year"].map(agr_land)

In [41]:
israel_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,4.21,107.6,29.9,92.59,82.7200,151.702500,80.5050,87.858333,355.5975,987.480833,4.308333,162.9000,0.94,19.44,21.91,-0.6,6.48,1.05
14,2021,4.28,105.1,29.7,92.67,87.6275,198.805000,94.0525,89.191667,395.5800,1082.482500,5.050000,176.6475,0.91,18.84,24.62,1.5,6.18,1.25
15,2022,4.47,109.4,NaN,92.76,92.3250,197.190833,109.8650,93.108333,441.1000,1264.530000,3.766667,207.1650,0.94,19.43,25.01,4.4,6.46,1.24
16,2023,4.57,NaN,NaN,92.85,105.8425,200.111667,114.3925,97.041667,469.5925,1378.055833,3.466667,214.2300,0.78,18.50,NaN,4.2,NaN,1.15
17,2024,NaN,NaN,NaN,NaN,126.7375,212.894167,113.4075,100.008333,499.5200,1448.846667,3.466667,224.5400,NaN,17.35,NaN,3.1,NaN,NaN


In [42]:
israel_df.to_parquet("../data/by_country/Israel_macro_profile.parquet", index=False)

In [43]:
malta_df = pd.read_parquet("../data/by_country/Malta_macro_profile.parquet")

In [45]:
malta_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,NaN,97.8,28.47,94.74,0.7075,0.772500,0.7450,106.368333,3.5925,10.327500,4.850000,141.5425,0.88,12.27,23.34,0.6,5.64,0.90
14,2021,NaN,95.6,27.34,94.81,0.7650,0.892500,0.9550,107.119167,4.1700,11.048333,3.766667,148.7375,0.84,12.02,23.82,1.5,5.36,0.85
15,2022,NaN,96.3,27.34,94.88,0.7925,1.121667,1.1075,113.685833,4.5650,11.829167,3.475000,158.6200,0.79,11.90,23.37,6.2,NaN,0.80
16,2023,NaN,NaN,NaN,94.94,0.8575,1.140000,0.9775,120.010833,5.1375,12.810000,3.483333,168.5050,0.70,11.81,NaN,5.1,NaN,0.40
17,2024,NaN,NaN,NaN,NaN,0.9600,1.320000,1.0325,122.940833,5.6150,13.795000,3.483333,179.3525,NaN,11.43,NaN,1.7,NaN,NaN


In [46]:
malta_labor_force = {
    2015: 68.8,
    2016: 70.6,
    2017: 72.1,
    2018: 74.7,
    2019: 75.8,
    2020: 77,
    2021: 78.2,
    2022: 79.8,
    2023: 80.4,
    2024: 80.9
}

In [47]:
malta_df["labor_force"] = malta_df["Year"].map(malta_labor_force)

In [48]:
malta_df.tail()

,Year,labor_force,livestock_prod,agr_land,urban_pop,gov_exp,reserves,investment,cpi,gdp,priv_cred,unemployment,house_price,rule_of_law_df,share_of_industry_df,tax_revenue_df,inflation_df,edu_spending_df,govermment_effective_df
13,2020,77.0,97.8,28.47,94.74,0.7075,0.772500,0.7450,106.368333,3.5925,10.327500,4.850000,141.5425,0.88,12.27,23.34,0.6,5.64,0.90
14,2021,78.2,95.6,27.34,94.81,0.7650,0.892500,0.9550,107.119167,4.1700,11.048333,3.766667,148.7375,0.84,12.02,23.82,1.5,5.36,0.85
15,2022,79.8,96.3,27.34,94.88,0.7925,1.121667,1.1075,113.685833,4.5650,11.829167,3.475000,158.6200,0.79,11.90,23.37,6.2,NaN,0.80
16,2023,80.4,NaN,NaN,94.94,0.8575,1.140000,0.9775,120.010833,5.1375,12.810000,3.483333,168.5050,0.70,11.81,NaN,5.1,NaN,0.40
17,2024,80.9,NaN,NaN,NaN,0.9600,1.320000,1.0325,122.940833,5.6150,13.795000,3.483333,179.3525,NaN,11.43,NaN,1.7,NaN,NaN


In [49]:
malta_df.to_parquet("../data/by_country/Malta_macro_profile.parquet", index=False)